# 01_extraccion
Extracción de clubes, equipos juveniles y movimientos desde Transfermarkt. Todo se guarda en `workspace.inferiores`.

**Orden:** 1) funciones comunes, 2) fase 1 (clubes), 3) fase 2 (equipos juveniles), 4) fase 3 (movimientos), 5) diagnóstico. Antes de todo corre el control del ciclo (`%run ./ciclo` y su compuerta) y al final marca la extracción como hecha.
La fase 4 (datos por jugador) NO está acá: la hace el notebook `02_jugadores` desde el Job `pipeline_inferiores`.

**Se puede correr entero y en orden.** Las páginas ya bajadas salen del caché del Volume (sin pedidos al sitio); solo lo que falta cuesta `DELAY` segundos por página. Las fases 2 y 3 son incrementales.

## Contenido

| Sección | Qué hace | Escribe |
|---|---|---|
| Control del ciclo | Abre el ciclo semanal si corresponde y sale si la extracción ya se hizo | `bronze_ciclos` |
| 0. Funciones comunes | Descarga con caché, freno y lectura de `robots.txt` | nada |
| Fase 1 | Clubes de cada liga y temporada | `bronze_clubs_pages`, `bronze_clubs` |
| Fase 2 | Equipos juveniles de cada club | `bronze_youth_teams_v2` |
| Fase 3 | Entradas y salidas de cada equipo en cada temporada | `bronze_arrivals`, `bronze_departures`, `bronze_transfers_status` |
| Diagnóstico | Resumen de los estados de la fase 3 | nada |
| Cierre | Marca la extracción del ciclo como hecha | `bronze_ciclos` |

## Reglas de acceso al sitio que aplica el código

- Descarga `robots.txt` de `www.transfermarkt.com` al inicio y, antes de cada pedido no cacheado, consulta si la ruta está permitida. Si no lo está, lanza `RuntimeError` y no hace el pedido.
- Espera 3 s (`DELAY`) antes de cada pedido no cacheado.
- Ante HTTP `202`, `403` o `429` lanza la excepción `Freno`. Ninguna fase la atrapa: la corrida se corta. No intenta resolver ningún desafío.
- Guarda cada página descargada en un Volume de Databricks. Una segunda corrida lee del Volume y no vuelve a pedir esas páginas.

## Control del ciclo
`%run ./ciclo` carga las funciones del notebook `ciclo` (estado del ciclo semanal en `workspace.inferiores.bronze_ciclos`) y su widget `FORZAR`.

La celda siguiente es la compuerta: llama a `abrir_ciclo_si_corresponde()`. Si el ciclo vigente ya tiene `extraccion_en` y `FORZAR` no vale `1`, termina el notebook con el texto `EXTRACCION YA HECHA (ciclo <lunes>)` y no hace pedidos. Si no, imprime el ciclo y sigue con la extracción.

In [ ]:
%run ./ciclo

In [ ]:
# Compuerta del ciclo: la extraccion corre una sola vez por ciclo
c = abrir_ciclo_si_corresponde()
if c["extraccion_en"] and not forzar():
    dbutils.notebook.exit(f"EXTRACCION YA HECHA (ciclo {c['lunes']})")
print(f"ciclo {c['lunes']} | abierto {c['inicio']} | extraccion pendiente")

## 0. Funciones comunes
Correr primero en cada sesión: las funciones y variables no persisten.

In [ ]:
import subprocess, re, time, hashlib, os, urllib.robotparser
from pyspark.sql import Row

TM = "https://www.transfermarkt.com"
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
CACHE = "/Volumes/workspace/inferiores/scraper/cache"
DELAY = 3.0   # segundos entre pedidos al sitio

class Freno(Exception):
    """El sitio devolvió 202, 403 o 429: hay que cortar la corrida, no seguir."""

def curl(url, out):
    r = subprocess.run(["curl", "-sL", "--max-time", "40", "-A", UA, "-w", "%{http_code}", "-o", out, url],
                       capture_output=True, text=True, timeout=60)
    return r.stdout.strip()

curl(TM + "/robots.txt", "/tmp/robots.txt")
rp = urllib.robotparser.RobotFileParser()
rp.parse(open("/tmp/robots.txt", encoding="utf-8", errors="replace").read().splitlines())

def get(url):
    p = f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}.html"
    if os.path.exists(p):
        return "200", open(p, encoding="utf-8", errors="replace").read()
    if not rp.can_fetch(UA, url):
        raise RuntimeError(f"robots.txt prohibe: {url}")
    time.sleep(DELAY)
    tmp = p + ".tmp"
    code = ""
    for attempt in range(3):
        code = curl(url, tmp)
        if code == "200":
            os.replace(tmp, p)
            return code, open(p, encoding="utf-8", errors="replace").read()
        if code in ("202", "403", "429"):
            raise Freno(f"HTTP {code} en {url}")
        if code == "404":
            return code, ""
        time.sleep(5 * (attempt + 1))
    return code or "0", ""

CLUB_RE = re.compile(r'href="/([^/"]+)/startseite/verein/(\d+)')

def etiqueta(title):
    m = re.search(r"(\d{2}/\d{2}|\d{4})\s*\|", title)
    return m.group(1) if m else ""

## Fase 1: clubes por temporada
Códigos de liga: `ARG1` (Primera, 1990 en adelante), `AR1N` (Primera, 1985-1989 y 2014-2023) y `ARG2` (Primera Nacional). Salen del notebook `00_codigos`.

La clave de temporada es el **título** de la página, no el `saison_id` (un año que el sitio no sirve devuelve la página por defecto con HTTP 200). Un título repetido dentro del mismo código es página de respaldo y se descarta.

Escribe `bronze_clubs_pages` (un pedido por fila) y `bronze_clubs` (club por código y temporada) con `overwrite`.

**Temporadas pedidas (`PLAN`)**

| Código | `saison_id` |
|---|---|
| `ARG1` | 2025, 2024 y de 2014 a 1990 |
| `AR1N` | de 2023 a 2014 y de 1989 a 1985 |
| `ARG2` | de 2025 a 2007 y 2000 |

Las dos consultas del final muestran la cantidad de clubes únicos, filas y temporadas de `bronze_clubs`, y las temporadas y clubes por código.

In [ ]:
PLAN = (
    [("ARG1", y) for y in [2025, 2024] + list(range(2014, 1989, -1))]
    + [("AR1N", y) for y in list(range(2023, 2013, -1)) + list(range(1989, 1984, -1))]
    + [("ARG2", y) for y in list(range(2025, 2006, -1)) + [2000]]
)

pages, clubs, vistos = [], [], set()
for code, y in PLAN:
    url = f"{TM}/-/startseite/wettbewerb/{code}/plus/?saison_id={y}"
    c, html = get(url)
    t = re.search(r"<title>([^<]*)</title>", html)
    title = t.group(1).strip() if t else ""
    found = sorted(set(CLUB_RE.findall(html)))
    dup = (code, title) in vistos
    vistos.add((code, title))
    pages.append(Row(code=code, saison_id=y, season_label=etiqueta(title), title=title,
                     http=int(c) if c.isdigit() else 0, n_clubs=len(found), descartada=dup))
    if not dup:
        clubs += [Row(code=code, season_label=etiqueta(title), club_id=cid, slug=s) for s, cid in found]
    print(code, y, c, len(found), "DESCARTADA" if dup else "", title[:50], flush=True)

spark.createDataFrame(pages).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.bronze_clubs_pages")
spark.createDataFrame(clubs).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.bronze_clubs")

display(spark.sql("""
  SELECT COUNT(DISTINCT club_id) AS clubes_unicos, COUNT(*) AS filas,
         COUNT(DISTINCT code, season_label) AS temporadas
  FROM workspace.inferiores.bronze_clubs"""))
display(spark.sql("""
  SELECT code, COUNT(DISTINCT season_label) AS temporadas, COUNT(DISTINCT club_id) AS clubes
  FROM workspace.inferiores.bronze_clubs GROUP BY code ORDER BY code"""))

## Fase 2: equipos juveniles por club
Por cada club de `bronze_clubs` se lee el `<ul>` del encabezado de su página (primer `<ul>` con `tiny_wappen` que contiene el link del propio club). Los ítems con otro `verein/{id}` son sus equipos (II, U20, U19, etc.). No se toman todos los links de la página: hay juveniles de otros clubes.

Escribe `bronze_youth_teams_v2` con `overwrite`. Columnas: `club_id`, `club_slug`, `team_id`, `team_slug`, `nombre` y `tipo`.

`tipo` lo calcula `tipo_de(nombre, slug)`:

| Valor | Regla |
|---|---|
| `II`, `III`, `U20`, `U19`, etc. | El nombre contiene `III`, `II` o `U` seguido de dos dígitos (con o sin guion) |
| `B` | El nombre termina en `B` |
| `JUVENILES` | El nombre o el `slug` contiene `jugend`, `juvenil` o `youth` |
| `OTRO` | Ninguna de las anteriores |

Las dos consultas del final muestran la cantidad de equipos por `tipo` y los equipos de tipo `OTRO` o `B` más los que tienen `Newell` en el nombre.

In [ ]:
from html import unescape

filas_clubes = spark.sql("""SELECT club_id, MIN(slug) AS slug FROM workspace.inferiores.bronze_clubs
                            GROUP BY club_id ORDER BY club_id""").collect()
ITEM = re.compile(r'<a[^>]*title="([^"]*)"[^>]*href="/([^/"]+)/startseite/verein/(\d+)"')

def bloque(page, cid):
    for m in re.finditer(r"(?s)<ul[^>]*>.*?</ul>", page):
        b = m.group(0)
        if "tiny_wappen" in b and f"/verein/{cid}\"" in b:
            return b
    return ""

def tipo_de(nombre, slug):
    n = unescape(nombre).strip()
    m = re.search(r"\b(III|II|U-?\d{2})\b", n)
    if m: return m.group(1).replace("-", "").upper()
    if re.search(r"\bB\s*$", n): return "B"
    low = (n + " " + slug).lower()
    if "jugend" in low or "juvenil" in low or "youth" in low: return "JUVENILES"
    return "OTRO"

nuevo = []
for r in filas_clubes:
    cid = str(r.club_id)
    page = get(f"{TM}/{r.slug}/startseite/verein/{cid}")[1]
    items = ITEM.findall(bloque(page, cid))
    own = [s for n, s, i in items if i == cid]
    club_slug = own[0] if own else r.slug
    for n, s, i in items:
        if i != cid:
            n = unescape(n)
            nuevo.append(Row(club_id=cid, club_slug=club_slug, team_id=i, team_slug=s, nombre=n, tipo=tipo_de(n, s)))

spark.createDataFrame(nuevo).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.bronze_youth_teams_v2")

display(spark.sql("""SELECT tipo, COUNT(*) AS n FROM workspace.inferiores.bronze_youth_teams_v2
                     GROUP BY tipo ORDER BY n DESC"""))
display(spark.sql("""SELECT club_slug, nombre, tipo FROM workspace.inferiores.bronze_youth_teams_v2
                     WHERE tipo IN ('OTRO','B') OR nombre LIKE '%Newell%'"""))

## Fase 3: movimientos por equipo y temporada
Para cada equipo de `bronze_youth_teams_v2` y cada temporada desde 1985 hasta el año en curso baja `/{team_slug}/transfers/verein/{team_id}/saison_id/{año}`.

Validación: el título tiene que traer `Transfers YY/YY` (año y año+1) y la página tiene que tener `<h2>Arrivals</h2>` y `<h2>Departures</h2>`; si no, el estado es `titulo_distinto` o `sin_h2` y no se guardan filas.

**Es incremental:** saltea los pares (equipo, temporada) que ya están en `bronze_transfers_status` con una temporada anterior a `DESDE` y sin estado `http_*` ni `error_*`, y vuelca cada 50 páginas, así que se puede cortar y retomar. Las temporadas desde `DESDE` (el año anterior al en curso) y los pares con `http_*` o `error_*` se vuelven a bajar en cada corrida: antes de escribir lo nuevo, `borrar_pares` borra lo anterior de esos pares. Si el sitio responde 202/403/429, la corrida **se corta** (la página que falló no se registra, así que se reintenta en la próxima).

La primera celda de código es una limpieza previa: borra de `bronze_arrivals` y `bronze_departures` los movimientos de pares (equipo, temporada) que no figuran en `bronze_transfers_status`. Si las tablas todavía no existen, lo avisa y sigue.

In [ ]:
# Limpieza previa (idempotente): borra movimientos de páginas que no llegaron a registrarse en el estado.
spark.sql("USE SCHEMA inferiores")
for t in ["bronze_arrivals", "bronze_departures"]:
    try:
        spark.sql(f"""DELETE FROM {t} WHERE NOT EXISTS (
            SELECT 1 FROM bronze_transfers_status s
            WHERE s.team_id = {t}.team_id AND s.season = {t}.season)""")
        print(t, "limpia")
    except Exception as e:
        print(t, "no existe todavía:", type(e).__name__)

### Fase 3: configuración, funciones y corrida

**Parámetros**

| Nombre | Contenido |
|---|---|
| `ANIO` | Año en curso |
| `TEMPORADAS` | De 1985 a `ANIO` |
| `DESDE` | `ANIO - 1`: las temporadas desde acá se vuelven a bajar en cada corrida |
| `LIMITE` | `None` para la corrida completa; un número limita cuántos pares se procesan |
| `LOTE` | `50`: cada cuántas páginas se escribe en las tablas |

**Funciones**

| Función | Qué hace |
|---|---|
| `partir(page)` | Separa el HTML en el bloque de entradas (desde `<h2>Arrivals</h2>`) y el de salidas (desde `<h2>Departures</h2>` hasta `Transfer record` o el final) |
| `filas(bloque)` | Extrae de cada fila: id y nombre del jugador, edad, id y nombre del otro club, `transfer_id`, `fee` y `date` |
| `fila_a_dict(...)` | Arma el diccionario de una fila con equipo, temporada y sentido |
| `procesar(team_id, slug, y)` | Descarga la página, valida el título y los `<h2>`, y devuelve el estado y las filas |
| `borrar_pares(pares)` | Borra de las tres tablas lo anterior de los pares (equipo, temporada) del lote |
| `volcar()` | Borra lo anterior del lote y escribe con `append` en `bronze_arrivals`, `bronze_departures` y `bronze_transfers_status` |

**Valores de `estado` en `bronze_transfers_status`**

| Estado | Significado |
|---|---|
| `ok` | La página pasó las validaciones |
| `http_<código>` | La descarga devolvió un código distinto de 200 |
| `titulo_distinto` | El título no trae `Transfers YY/YY` de esa temporada |
| `sin_h2` | Falta `<h2>Arrivals</h2>` o `<h2>Departures</h2>` |
| `error_<excepción>` | Falló el procesamiento de esa página |

La columna `date` se conserva por compatibilidad con las tablas existentes pero no es confiable (el código documenta que capta un texto secundario de la celda de `fee`, vacío en el 80,9 % de las filas). La fecha de referencia es `season`.

**Corrida:** recorre los pares pendientes e imprime el avance cada `LOTE` páginas. Si aparece `Freno`, guarda lo pendiente del lote, imprime `CORTADA POR FRENO` y al final lanza `RuntimeError`, de modo que las celdas siguientes (incluido el cierre del ciclo) no se ejecutan.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import datetime

spark.sql("USE SCHEMA inferiores")

COL_ID, COL_SLUG = "team_id", "team_slug"
ANIO = datetime.date.today().year
TEMPORADAS = list(range(1985, ANIO + 1))
DESDE = ANIO - 1     # las temporadas >= DESDE se vuelven a bajar en cada corrida
LIMITE = None        # None para la corrida completa
LOTE = 50

# La columna `date` se conserva por compatibilidad con las tablas existentes, pero capta un texto
# secundario de la celda de `fee` (80,9 % vacía): no usarla. La fecha de referencia es `season`.
ESQ_MOV = StructType([
    StructField("team_id", StringType()),
    StructField("team_slug", StringType()),
    StructField("season", IntegerType()),
    StructField("sentido", StringType()),
    StructField("player_id", StringType()),
    StructField("player_name", StringType()),
    StructField("age", StringType()),
    StructField("counterpart_id", StringType()),
    StructField("counterpart_name", StringType()),
    StructField("transfer_id", StringType()),
    StructField("fee", StringType()),
    StructField("date", StringType()),
])
ESQ_EST = StructType([
    StructField("team_id", StringType()),
    StructField("season", IntegerType()),
    StructField("estado", StringType()),
    StructField("n_arrivals", IntegerType()),
    StructField("n_departures", IntegerType()),
])

def partir(page):
    a = re.search(r"<h2[^>]*>\s*Arrivals\s*</h2>", page)
    d = re.search(r"<h2[^>]*>\s*Departures\s*</h2>", page)
    r = re.search(r"Transfer record", page)
    return page[a.start():d.start()], page[d.start():(r.start() if r else len(page))]

def filas(bloque):
    out = []
    patron = r'<tr class="(?:odd|even)">(.*?)</tr>\s*(?=<tr class="(?:odd|even)">|</tbody>)'
    for tr in re.findall(patron, bloque, flags=re.S):
        pl = re.search(r'href="/[^"]*/profil/spieler/(\d+)"[^>]*>([^<]*)</a>', tr)
        cl = re.search(r'href="/[^"]*/startseite/verein/(\d+)"[^>]*>([^<]*)</a>\s*</td>', tr)
        tf = re.search(r'/transfer_id/(\d+)">(.*?)</a>', tr, flags=re.S)
        edad = re.search(r'<td class="zentriert">(\d+)</td>', tr)
        fee = None
        fecha = None
        if tf:
            fee = re.sub(r"<[^>]+>", "|", tf.group(2)).split("|")[0].strip()
            fe = re.search(r'<i class="normaler-text">([^<]*)</i>', tf.group(2))
            if fe:
                fecha = fe.group(1).strip()
        out.append((
            pl.group(1) if pl else None,
            pl.group(2).strip() if pl else None,
            edad.group(1) if edad else None,
            cl.group(1) if cl else None,
            cl.group(2).strip() if cl else None,
            tf.group(1) if tf else None,
            fee,
            fecha,
        ))
    return out

def fila_a_dict(f, team_id, slug, y, sentido):
    return dict(team_id=str(team_id), team_slug=slug, season=y, sentido=sentido,
                player_id=f[0], player_name=f[1], age=f[2],
                counterpart_id=f[3], counterpart_name=f[4],
                transfer_id=f[5], fee=f[6], date=f[7])

def procesar(team_id, slug, y):
    c, page = get(f"{TM}/{slug}/transfers/verein/{team_id}/saison_id/{y}")
    if str(c) != "200":
        return f"http_{c}", 0, 0, [], []
    t = re.search(r"<title>([^<]*)</title>", page)
    titulo = t.group(1) if t else ""
    esperado = f"Transfers {str(y)[2:]}/{str(y + 1)[2:]}"
    if esperado not in titulo:
        return "titulo_distinto", 0, 0, [], []
    if not (re.search(r"<h2[^>]*>\s*Arrivals\s*</h2>", page) and re.search(r"<h2[^>]*>\s*Departures\s*</h2>", page)):
        return "sin_h2", 0, 0, [], []
    arr, dep = partir(page)
    fa, fd = filas(arr), filas(dep)
    return "ok", len(fa), len(fd), fa, fd

spark.sql("""CREATE TABLE IF NOT EXISTS bronze_transfers_status
             (team_id STRING, season INT, estado STRING, n_arrivals INT, n_departures INT)""")

# Pares ya hechos: temporadas cerradas y sin error. Las recientes y los errores se vuelven a bajar.
hechos = {(r.team_id, r.season) for r in spark.table("bronze_transfers_status")
          .where(f"season < {DESDE} AND estado NOT LIKE 'http_%' AND estado NOT LIKE 'error_%'").collect()}

equipos = [(str(r[COL_ID]), r[COL_SLUG]) for r in
           spark.table("bronze_youth_teams_v2").select(COL_ID, COL_SLUG).distinct().collect()]
pendientes = [(t, s, y) for (t, s) in equipos for y in TEMPORADAS if (t, y) not in hechos]
if LIMITE:
    pendientes = pendientes[:LIMITE]
print(f"equipos: {len(equipos)} | refresco desde temporada {DESDE} | cerradas ya hechas: {len(hechos)} | a procesar ahora: {len(pendientes)}")

bufer_arr, bufer_dep, bufer_est = [], [], []

def borrar_pares(pares):
    # Borra lo anterior de los pares del lote justo antes de escribir lo nuevo.
    if not pares:
        return
    cond = " OR ".join(f"(team_id = '{t}' AND season = {y})" for t, y in pares)
    for tabla in ["bronze_arrivals", "bronze_departures", "bronze_transfers_status"]:
        if spark.catalog.tableExists(tabla):
            spark.sql(f"DELETE FROM {tabla} WHERE {cond}")

def volcar():
    global bufer_arr, bufer_dep, bufer_est
    borrar_pares([(r.team_id, r.season) for r in bufer_est])
    if bufer_arr:
        spark.createDataFrame(bufer_arr, ESQ_MOV).write.mode("append").saveAsTable("bronze_arrivals")
    if bufer_dep:
        spark.createDataFrame(bufer_dep, ESQ_MOV).write.mode("append").saveAsTable("bronze_departures")
    if bufer_est:
        spark.createDataFrame([tuple(r) for r in bufer_est], ESQ_EST).write.mode("append").saveAsTable("bronze_transfers_status")
    bufer_arr, bufer_dep, bufer_est = [], [], []

inicio = time.time()
cortada = None
try:
    for i, (team_id, slug, y) in enumerate(pendientes, 1):
        try:
            estado, na, nd, fa, fd = procesar(team_id, slug, y)
        except Freno:
            raise   # corta la corrida; esta página no se registra y se reintenta después
        except Exception as e:
            estado, na, nd, fa, fd = f"error_{type(e).__name__}", 0, 0, [], []
        bufer_arr += [fila_a_dict(f, team_id, slug, y, "arrival") for f in fa]
        bufer_dep += [fila_a_dict(f, team_id, slug, y, "departure") for f in fd]
        bufer_est.append(Row(team_id=team_id, season=y, estado=estado, n_arrivals=na, n_departures=nd))
        if i % LOTE == 0:
            volcar()
            el = time.time() - inicio
            print(f"{i}/{len(pendientes)} | {el/60:.1f} min | restante aprox {(el/i)*(len(pendientes)-i)/60:.0f} min", flush=True)
    print("listo")
except Freno as e:
    cortada = e
    print("CORTADA POR FRENO:", e, "| volver a correr más tarde; retoma desde acá.")
finally:
    volcar()

if cortada:
    raise RuntimeError(f"Fase 3 incompleta por freno: {cortada}")

## Diagnóstico del estado de la fase 3
Todo lo que no sea `ok` merece una mirada. `sin_h2` y `titulo_distinto` no se reintentan en las temporadas anteriores a `DESDE`; `http_*` y `error_*` sí, borrándolos primero (ver la fase 3).

La primera consulta cuenta páginas, entradas y salidas por estado. La segunda lista hasta 50 filas con estado distinto de `ok`.

In [ ]:
spark.sql("USE SCHEMA inferiores")
display(spark.sql("SELECT estado, count(*) n, sum(n_arrivals) llegadas, sum(n_departures) salidas FROM bronze_transfers_status GROUP BY estado ORDER BY n DESC"))
display(spark.sql("SELECT * FROM bronze_transfers_status WHERE estado <> 'ok' ORDER BY estado, team_id, season LIMIT 50"))

## Cierre de la extracción del ciclo
`marcar("extraccion_en")` guarda la hora actual en `bronze_ciclos` para el ciclo vigente, solo si ese campo estaba vacío. Con eso, `02_jugadores` puede empezar a bajar los jugadores del ciclo. Si la fase 3 se cortó por freno, el notebook ya terminó con `RuntimeError` y esta celda no se ejecuta.

In [ ]:
marcar("extraccion_en")

## Fase 4: datos por jugador
**No se hace en este notebook.** La corre el notebook `02_jugadores` desde el Job `pipeline_inferiores` (cuota diaria, pausas, `bronze_corridas`, refresco de activos). No correrla a mano mientras el Job está activo: escribiría filas duplicadas en `bronze_player_raw`.